# Projeto (Seção 8) — solução comentada

Continua a solução comentada da seção 7. Enunciado em [`07_projeto`](07_projeto.ipynb).

Esta seção tem dois achados importantes em relação à solução do instrutor:
1. **`TypeError` com `registration_date` ausente** (o validador não trata `None` e `validate_default=True` o executa no default);
2. **atribuição pode deixar o modelo inconsistente**: a regra "registro ≥ fabricação" só é checada num dos dois campos.

## 1. "Banco" de países e o tipo `Country`

In [1]:
from datetime import date
from enum import Enum
from typing import Annotated, Any, TypeVar
from uuid import UUID, uuid4
from pydantic import (UUID4, AfterValidator, BaseModel, ConfigDict, Field, ValidationError, ValidationInfo,
                      field_serializer, field_validator)
from pydantic.alias_generators import to_camel

countries = {
    "australia": ("Australia", "AUS"),
    "canada": ("Canada", "CAN"),
    "china": ("China", "CHN"),
    "france": ("France", "FRA"),
    "germany": ("Germany", "DEU"),
    "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"),
    "norway": ("Norway", "NOR"),
    "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"),
    "sanmarino": ("San Marino", "SMR"),
    "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"),
    "united kingdom": ("United Kingdom", "GBR"),
    "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"),
    "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"),
    "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"),
}
countries["brazil"] = countries["brasil"] = ("Brazil", "BRA")       # sugestão do enunciado: incluir o próprio país

VALID_COUNTRY_KEYS = sorted(countries)                              # calculado uma vez, fora do validador


def lookup_country(name: str) -> tuple[str, str]:
    key = " ".join(name.split()).casefold()        # strip + espaços internos colapsados + casefold
    try:
        return countries[key]
    except KeyError:
        raise ValueError(f"unknown country {name!r}; expected one of: {', '.join(VALID_COUNTRY_KEYS)}") from None


Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]

Testando o tipo isoladamente (com `TypeAdapter`, como na seção 7):

In [2]:
from pydantic import TypeAdapter

CountryAdapter = TypeAdapter(Country)
for raw in ["UK", "Uk", "uk ", "  united   kingdom ", "Brasil", "usa"]:
    print(f"{raw!r}: {CountryAdapter.validate_python(raw)}")
try:
    CountryAdapter.validate_python("Lunar Colony")
except ValidationError as ex:
    print(f"'Lunar Colony': {ex.errors()[0]['msg'][:80]}...")

'UK': United Kingdom
'Uk': United Kingdom
'uk ': United Kingdom
'  united   kingdom ': United Kingdom
'Brasil': Brazil
'usa': United States of America
'Lunar Colony': Value error, unknown country 'Lunar Colony'; expected one of: australia, brasil,...


## 2. Modelo

A regra "registro não pode ser anterior à fabricação" é validada **nos dois campos**:

- em `registration_date`: na criação (o campo vem depois de `manufactured_date`, então `info.data` já o tem) e na atribuição;
- em `manufactured_date`: só tem efeito na **atribuição** — é o caso em que `info.data` já contém `registration_date`.

Assim, nenhuma ordem de atribuição consegue deixar a instância inconsistente. Os dois validadores tratam `None`.

In [3]:
class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


T = TypeVar("T")
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

    id_: UUID4 = Field(default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD")
    top_features: BoundedList[BoundedString] | None = None
    vin: BoundedString
    number_of_doors: int = Field(default=4, validation_alias="doors", ge=2, le=4, multiple_of=2)
    registration_country: Country | None = None
    registration_date: date | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", "registration_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

    @field_validator("manufactured_date")
    @classmethod
    def manufactured_not_after_registration(cls, value: date, info: ValidationInfo) -> date:
        registered = info.data.get("registration_date")           # só existe aqui em atribuições
        if registered is not None and value > registered:
            raise ValueError(f"manufactured_date {value} is after registration_date {registered}")
        return value

    @field_validator("registration_date")
    @classmethod
    def registration_not_before_manufacture(cls, value: date | None, info: ValidationInfo) -> date | None:
        manufactured = info.data.get("manufactured_date")
        if value is not None and manufactured is not None and value < manufactured:
            raise ValueError(f"registration_date {value} is before manufactured_date {manufactured}")
        return value

## 3. Testes do enunciado

In [4]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'United States of America',
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB'
}

expected_json_by_alias = '{"id":"c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7","manufacturer":"BMW","seriesName":"M4 Competition xDrive","type":"Convertible","isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,"topFeatures":["6 cylinders","all-wheel drive","convertible"],"vin":"1234567890","numberOfDoors":2,"registrationCountry":"United States of America","registrationDate":"2023/06/01","licensePlate":"AAA-BBB"}'

In [5]:
car = Automobile.model_validate(data)
assert car.model_dump(by_alias=True) == expected_by_alias
assert car.model_dump_json(by_alias=True) == expected_json_by_alias
fields = list(Automobile.model_fields)
assert fields.index("registration_date") == fields.index("registration_country") + 1
print("testes do enunciado: ok")

testes do enunciado: ok


## 4. Testes extras

In [6]:
def errs(**changes) -> list[tuple]:
    try:
        Automobile.model_validate({**data, **changes})
    except ValidationError as ex:
        return [(err["loc"], err["type"], err["msg"][:60]) for err in ex.errors()]
    return []

# o caso que derruba a solução do instrutor: sem registrationDate
no_reg = {k: v for k, v in data.items() if k != "registrationDate"}
print(f"sem registrationDate: {Automobile.model_validate(no_reg).registration_date}")
print(f"registrationDate=None: {errs(registrationDate=None)}")

# fronteira: registro no mesmo dia da fabricação é permitido
print(f"registrationDate == completionDate: {errs(registrationDate='2023-01-01')}")

# os dois erros do enunciado reportados juntos
for e in errs(registrationCountry="Lunar Colony", registrationDate="2022-06-01"):
    print(f"erro: {e}")

# variações de escrita do país
for raw in ("UK", " great   britain ", "USA"):
    print(f"registrationCountry={raw!r}: {Automobile.model_validate({**data, 'registrationCountry': raw}).registration_country}")

# JSON com registration_date None (o serializador é json-unless-none)
import json
print(f"json['registrationDate'] sem registro: {json.loads(Automobile.model_validate(no_reg).model_dump_json(by_alias=True))['registrationDate']}")
print("testes extras: ok")

sem registrationDate: None
registrationDate=None: []
registrationDate == completionDate: []
erro: (('registrationCountry',), 'value_error', "Value error, unknown country 'Lunar Colony'; expected one of")
erro: (('registrationDate',), 'value_error', 'Value error, registration_date 2022-06-01 is before manufact')
registrationCountry='UK': United Kingdom
registrationCountry=' great   britain ': United Kingdom
registrationCountry='USA': United States of America
json['registrationDate'] sem registro: None
testes extras: ok


## 5. Consistência na atribuição (o achado 2)

Com a regra só em `registration_date` (como no instrutor), alterar `manufactured_date` para depois do registro **passa sem erro**:

In [7]:
class AutomobileOneSided(Automobile):
    @field_validator("manufactured_date")
    @classmethod
    def manufactured_not_after_registration(cls, value: date, info: ValidationInfo) -> date:
        return value                                   # desliga a checagem do lado da fabricação

one_sided = AutomobileOneSided.model_validate(data)
one_sided.manufactured_date = date(2024, 1, 1)
print(f"um lado só — aceito e inconsistente: fabricado {one_sided.manufactured_date}, registrado {one_sided.registration_date}")

car = Automobile.model_validate(data)
try:
    car.manufactured_date = date(2024, 1, 1)
except ValidationError as ex:
    print(f"dois lados — rejeitado: {ex.errors()[0]['msg']}")
print(f"instância intacta após a falha: fabricado {car.manufactured_date}, registrado {car.registration_date}")

um lado só — aceito e inconsistente: fabricado 2024-01-01, registrado 2023-06-01
dois lados — rejeitado: Value error, manufactured_date 2024-01-01 is after registration_date 2023-06-01
instância intacta após a falha: fabricado 2023-01-01, registrado 2023-06-01


> ⚠️ Por que não `@model_validator(mode="after")`? Ele também levanta erro na atribuição, **mas o valor já foi gravado** antes da
> checagem — a instância fica inconsistente mesmo após a exceção (testado no Pydantic 2.13). Validadores de campo rejeitam **antes** de gravar.

In [8]:
from pydantic import model_validator

class AutomobileModelValidator(AutomobileOneSided):
    @model_validator(mode="after")
    def dates_consistent(self):
        if self.registration_date is not None and self.registration_date < self.manufactured_date:
            raise ValueError("registration_date before manufactured_date")
        return self

mv = AutomobileModelValidator.model_validate(data)
try:
    mv.manufactured_date = date(2024, 1, 1)
except ValidationError as ex:
    print(f"model_validator — erro: {ex.errors()[0]['msg']}")
print(f"model_validator — estado após a falha: fabricado {mv.manufactured_date} (já gravado!)")

model_validator — erro: Value error, registration_date before manufactured_date
model_validator — estado após a falha: fabricado 2024-01-01 (já gravado!)


## 6. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| `Country` | `AfterValidator` + `casefold` | idem, + colapsa espaços internos, mensagem com o valor recebido, Brasil incluído |
| `registration_date` ausente | **`TypeError`** (validador não trata `None`; `validate_default` o executa) | tratado |
| regra de datas | só em `registration_date` | nos **dois** campos → atribuição nunca deixa inconsistente |
| `id_` | `UUID4 \| None` (regressão) | `UUID4` |
| testes | caminho feliz + 1 caso inválido | + ausência, `None`, fronteira, variações de país, consistência na atribuição |